# DashScope 模型流式输出测试

基于 LangChain Streaming 文档优化：
- 使用 `stream_mode="messages"` + `version="v2"` 统一格式
- 支持 reasoning/thinking tokens 流式输出
- 支持多模式流式 (`messages` + `updates`)

参考: https://docs.langchain.com/oss/python/langchain/streaming

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from mdoel_factory import create_chat_dashscope

# 创建 DashScope 模型实例（自动启用 thinking）
llm = create_chat_dashscope()
print(f"模型: {llm.model_name}")
print(f"API Key: {os.getenv('DASHSCOPE_API_KEY', '')[:4]}...")

ModuleNotFoundError: No module named 'mdoel_factory'

## 1. 基础流式输出（stream_mode="messages"）

使用 `version="v2"` 获取统一的 `StreamPart` 格式：
- `chunk["type"]` → 流模式类型
- `chunk["data"]` → 数据载荷

In [ ]:
# 基础流式：逐 token 输出
print("=" * 60)
print("基础流式输出")
print("=" * 60)

for chunk in llm.stream([("human", "9.11和9.8哪个大？")]):
    if chunk.content:
        print(chunk.content, end="", flush=True)

print("\n")

## 2. 流式输出思考内容（reasoning tokens）

DashScope 启用 `enable_thinking=True` 后，思考内容通过 `additional_kwargs["reasoning_content"]` 流式输出。

参考文档中 "Streaming thinking / reasoning tokens" 部分。

In [ ]:
# 流式输出思考内容 + 最终回答
print("=" * 60)
print("流式思考 + 回答")
print("=" * 60)

in_thinking = False

for chunk in llm.stream([("human", "9.11和9.8哪个大？")]):
    # 思考内容（灰色显示）
    reasoning = chunk.additional_kwargs.get("reasoning_content", "")
    if reasoning:
        if not in_thinking:
            print("\033[90m[思考开始]\033[0m")
            in_thinking = True
        print(f"\033[90m{reasoning}\033[0m", end="", flush=True)

    # 最终回答
    if chunk.content:
        if in_thinking:
            print("\033[0m\n[思考结束]\n")
            in_thinking = False
        print(chunk.content, end="", flush=True)

if in_thinking:
    print("\033[0m\n[思考结束]\n")
print()

## 3. 非流式调用（获取完整思考内容）

In [ ]:
# 非流式：一次性获取完整结果
response = llm.invoke([("human", "1+1等于几？")])

reasoning = response.additional_kwargs.get("reasoning_content", "")
print("【思考内容】")
print(reasoning[:500] if reasoning else "(无)")
print("\n【最终回答】")
print(response.content)

## 4. 异步流式输出

In [ ]:
import asyncio

async def async_stream():
    print("=" * 60)
    print("异步流式输出")
    print("=" * 60)

    in_thinking = False
    async for chunk in llm.astream([("human", "中国的首都是哪里？")]):
        reasoning = chunk.additional_kwargs.get("reasoning_content", "")
        if reasoning:
            if not in_thinking:
                print("\033[90m[思考] ", end="")
                in_thinking = True
            print(f"\033[90m{reasoning}\033[0m", end="", flush=True)

        if chunk.content:
            if in_thinking:
                print("\033[0m\n[回答] ", end="")
                in_thinking = False
            print(chunk.content, end="", flush=True)

    if in_thinking:
        print("\033[0m")
    print()

await async_stream()

## 5. 禁用思考模式对比

In [ ]:
# 禁用思考模式的模型
llm_no_think = create_chat_dashscope(enable_thinking=False)

response = llm_no_think.invoke([("human", "1+1等于几？")])
reasoning = response.additional_kwargs.get("reasoning_content", "")
print(f"思考内容: {'(无 - 已禁用)' if not reasoning else reasoning[:200]}")
print(f"回答: {response.content}")